<a href="https://colab.research.google.com/github/vaidiknakrani/parul_AI_ML_Learning/blob/main/Day_9_GenAI_document_q_a_mini_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Document Q&A (mini RAG) — Kaggle, no API key

Ask questions about one or more PDF files using entirely local, open-source models. The notebook creates embeddings with `all-MiniLM-L6-v2`, searches them with FAISS, and produces an answer with `Qwen/Qwen2.5-1.5B-Instruct`.

**No OpenAI, Hugging Face, or other API key is required.** Enable Internet only for the first run so Kaggle can download packages and public model weights. After saving the models as a Kaggle Dataset, you can run with Internet disabled.

## Kaggle setup

1. Create a new Kaggle Notebook and turn on a **GPU** accelerator (T4 is enough).
2. Create a Kaggle Dataset containing your PDF files, then attach it through **Add Input**.
3. Turn on Internet for this first run.
4. Run every cell in order. In the next cell, replace `YOUR_DATASET_FOLDER` with the folder name shown under `/kaggle/input/`.

Do not upload confidential documents to a public Kaggle dataset.

In [ ]:
# Install the small libraries not guaranteed to be preinstalled in every Kaggle image.
!pip -q install -U sentence-transformers faiss-cpu pypdf accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 3.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 49.8 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 22.0 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import re
import numpy as np
import torch
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

# Change this to your attached Kaggle Dataset folder.
DOCUMENT_FOLDER = Path('/kaggle/input/YOUR_DATASET_FOLDER')
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
LLM_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
TOP_K = 4

assert torch.cuda.is_available(), 'Enable GPU in Kaggle: Settings → Accelerator → GPU.'
print('GPU:', torch.cuda.get_device_name(0))

AssertionError: Enable GPU in Kaggle: Settings → Accelerator → GPU.

## Read and chunk the PDFs

A chunk is a small overlapping piece of text. The overlap prevents important context from being split between chunks.

In [ ]:
def extract_pdf_pages(pdf_path):
    reader = PdfReader(str(pdf_path))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        text = re.sub(r'\s+', ' ', text).strip()
        if text:
            pages.append({'source': pdf_path.name, 'page': page_number, 'text': text})
    return pages

def chunk_text(text, chunk_size=900, overlap=160):
    words = text.split()
    step = chunk_size - overlap
    return [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), step) if words[i:i + chunk_size]]

pdf_files = sorted(DOCUMENT_FOLDER.rglob('*.pdf'))
if not pdf_files:
    raise FileNotFoundError(f'No PDFs found in {DOCUMENT_FOLDER}. Check DOCUMENT_FOLDER and attach your dataset.')

chunks = []
for pdf_file in pdf_files:
    for page_data in extract_pdf_pages(pdf_file):
        for text_chunk in chunk_text(page_data['text']):
            chunks.append({**page_data, 'text': text_chunk})

print(f'Loaded {len(pdf_files)} PDF(s) and created {len(chunks)} chunks.')
print('Example:', chunks[0]['text'][:300])

## Create the searchable vector index

The embedding model converts each chunk into numbers. FAISS finds chunks that are closest in meaning to a question.

In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device='cuda')
embeddings = embedder.encode(
    [item['text'] for item in chunks],
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=64
).astype('float32')

index = faiss.IndexFlatIP(embeddings.shape[1])  # inner product = cosine similarity after normalization
index.add(embeddings)
print(f'FAISS index contains {index.ntotal} chunks.')

## Load the local language model

This downloads a public 1.5B-parameter model on the first run. It runs locally in the Kaggle session; your question is not sent to an API.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
).eval()
print('Model loaded.')

In [ ]:
def retrieve(question, top_k=TOP_K):
    query = embedder.encode([question], normalize_embeddings=True).astype('float32')
    scores, ids = index.search(query, top_k)
    return [(chunks[i], float(score)) for i, score in zip(ids[0], scores[0])]

def answer_question(question, top_k=TOP_K, max_new_tokens=350):
    results = retrieve(question, top_k)
    context = '\n\n'.join(
        f'[{n}] Source: {item["source"]}, page {item["page"]}\n{item["text"]}'
        for n, (item, _) in enumerate(results, start=1)
    )
    messages = [
        {'role': 'system', 'content': 'Answer only from the supplied context. If the answer is not in the context, say: I could not find that in the documents. Cite sources like [1] or [2].'},
        {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {question}'}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([prompt], return_tensors='pt').to(model.device)
    with torch.inference_mode():
        generated = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    answer_tokens = generated[0][model_inputs.input_ids.shape[1]:]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()
    print('Answer:\n', answer)
    print('\nRetrieved passages:')
    for n, (item, score) in enumerate(results, start=1):
        print(f'[{n}] {item["source"]}, page {item["page"]} (similarity {score:.3f})')
    return answer, results

# Replace with a question about your documents.
answer_question('What are the most important points in these documents?')

## Optional: ask more questions

Run the next cell repeatedly with a new question. The model and index stay in memory, so later answers are faster.

### Troubleshooting
- **Out of memory:** change `LLM_MODEL` to `Qwen/Qwen2.5-0.5B-Instruct` or reduce `TOP_K` to 3.
- **No PDFs found:** correct `DOCUMENT_FOLDER`; the exact folder name appears under `/kaggle/input`.
- **Scanned PDF gives blank text:** OCR it before using this notebook, because `pypdf` reads embedded text rather than images.
- **Internet-off run:** save the model folders as a private Kaggle Dataset and set model paths to `/kaggle/input/<your-model-dataset>/<folder>`.

In [ ]:
question = 'Write your next question here'
# answer_question(question)